# Tool creation methods in smolagents. How?
1. @tool decorator - recommended way to define simple tools.
2. creating subclass of tool class  - for more complex functionality
3. Using the smolagents default toolbox
4. Importing a tool from the hugging face hub
5. Importing huggingface space as a tool
6. Importing a langchain tool
7. Importing a tool collection from any MCP server

* You can also share your tool to the huggingface hub
   
* **Here’s how a simple search tool interface might look:**
1. Name: What the tool is called
2. Tool description: What the tool does
3. Input types and descriptions: What arguments the tool accepts
4. Output type: What the tool returns

## 1. @Tool decorator
* **smolagents will parse basic information about the function from Python. So if you name your function clearly and write a good docstring, it will be easier for the LLM to use.**

Using this approach, we define a function with:
* A clear and descriptive function name that helps the LLM understand its purpose.
* 
Type hints for both inputs and outputs to ensure proper usage
* 
A detailed description, including an Args: section where each argument is explicitly described. These descriptions provide valuable context for the LLM, so it’s important to write them carefully.



In [21]:
!pip install smolagents

In [1]:
#LET THE AGENT FIND THE BEST CATERING SERVICES from a list IN GOTHAM based best rating
from smolagents import CodeAgent, LiteLLMModel, tool

model = LiteLLMModel(
    model_id="ollama/qwen2.5-coder:1.5b",
    api_base="http://localhost:11434"
)
@tool
def catering_service_tool(query:str) ->str:
    #typestrings
    """
    Returns the highest-rated catering service in Gotham City.
    
    Args:
        query: Search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service
agent = CodeAgent(tools=[catering_service_tool],
                  model=model)

# Run the agent to find the best catering service
agent.run("Use the catering_service_tool to find the highest-rated catering service in Gotham City.")

print(result)



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the catering_service_tool to find the highest-rated catering service in Gotham City.                        │
│                                                                                                                 │
╰─ LiteLLMModel - ollama/qwen2.5-coder:1.5b ──────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = catering_service_tool(query="Gotham City")                                                              
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Catering Co.

Out: None

[Step 1: Duration 39.48 seconds| Input tokens: 2,100 | Output tokens: 63]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (86 outputs left out)

[Step 20: Duration 73.76 seconds| Input tokens: 47,427 | Output tokens: 2,901]

Reached max steps.

[Step 21: Duration 82.84 seconds| Input tokens: 49,477 | Output tokens: 3,180]

NameError: name 'result' is not defined

## 2. Defining a Tool as a Python Class

* name: The tool’s name.* 
description: A description used to populate the agent’s system prompt.* 
inputs: A dictionary with keys type and description, providing information to help the Python interpreter process inputs* .
output_type: Specifies the expected output typ* e.
forward: The method containing the inference logic to execute.

In [21]:
from smolagents import Tool, CodeAgent, LiteLLMModel

# Set your Groq key
import os
os.environ["GROQ_API_KEY"] = "xxx"

# Connect to Groq using the groq/ provider prefix
model_g = LiteLLMModel(
    model_id="groq/openai/gpt-oss-120b", # Or another supported model
    temperature=0.1
)

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {#key is category, value is anther dictionary in the bracket
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = { #inputs is category:{type: description}
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

# Instantiate the tool
party_theme_tool = SuperheroPartyThemeTool()
agent = CodeAgent(tools=[party_theme_tool],
                  model=model_g)

# Run the agent to generate a party theme idea
result = agent.run(
    "What would be a good superhero party idea for a 'villain masquerade' theme?"
)

print(result)  # Output: "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What would be a good superhero party idea for a 'villain masquerade' theme?                                     │
│                                                                                                                 │
╰─ LiteLLMModel - groq/openai/gpt-oss-120b ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

[Step 1: Duration 1.34 seconds| Input tokens: 2,120 | Output tokens: 237]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme = superhero_party_theme_generator(category="villain masquerade")                                           
  final_answer(theme)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

[Step 2: Duration 0.61 seconds| Input tokens: 4,298 | Output tokens: 362]

Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.


## Default Toolbox
smolagents comes with a set of pre-built utilities/tools that can be directly injected into your agent. The default toolbox includes:

* PythonInterpreterTool
* FinalAnswerTool
* UserInputTool
* DuckDuckGoSearchTool
* GoogleSearchTool
* VisitWebpageTool

## Sharing and importing a tool from the hub

**Upload a tool using the push_to_hub() method, import using the load_tool() function**

In [ ]:
#eg: Alfred can share his catering_service_tool to help others find the best catering services in Gotham
party_theme_tool.push_to_hub("{your_username}/catering_service_tool", 
                             token="<YOUR_HUGGINGFACEHUB_API_TOKEN>")

In [ ]:
pip install litellm

In [ ]:
'''
Alfred might want to generate a promotional image for the party using AI. Instead of building a tool from scratch, 
he can leverage a predefined one from the community:
'''

from smolagents import load_tool, CodeAgent, LiteLLMModel
#importing an image generation tool from the hub
image_generation_tool = load_tool(
    "m-ric/text-to-image",
    trust_remote_code=True
)

model = LiteLLMModel(
    model_id="ollama/qwen2.5-coder:1.5b",
    api_base="http://localhost:11434"
)
agent = CodeAgent(
    tools=[image_generation_tool],
    model=model
)

agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")


## Importing a Hugging Face Space as a Tool

**This opens up possibilities for integrating with thousands of spaces from the community for tasks from image generation to data analysis.**

In [ ]:
!pip install gradio_client

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, Tool

image_generation_tool = Tool.from_space(
    "black-forest-labs/FLUX.1-schnell",
    name="image_generator",
    description="Generate an image from a prompt"
)

model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")

agent = CodeAgent(tools=[image_generation_tool], model=model)

agent.run(
    "Improve this prompt, then generate an image of it.",
    additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
)

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

image_path = '/tmp/gradio/d5a8dfbade97e9b9d99f78d5ccaa73db6d4b8dc428662f2f25bde1de1bd77b81/image.webp'

img = PILImage.open(image_path)
img

## Importing a LangChain Tool
**LangChain is a massive, popular framework for building applications with Large Language Models (LLMs)**
* It provides hundreds of pre-built integrations, utilities, and connectors to external services, databases, and APIs (like Google Search, Wikipedia, calculators, and SQL databases)
* While Hugging Face has many great tools, LangChain has been around longer and has an enormous library of pre-integrated third-party tools (APIs for weather, financial data, specialized databases, and web search engines). If a specific tool doesn't exist natively in smolagents or on the Hugging Face Hub, you can almost certainly find it in LangChain.

In [25]:
'''
 Alfred, ever the perfectionist, is preparing for a spectacular superhero night at
 Wayne Manor while the Waynes are away. 
 To make sure every detail exceeds expectations, he taps into LangChain tools to find 
 top-tier entertainment ideas.
'''
!pip install langchain-community google-search-results

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
    --------------------------------------- 0.0/2.4 MB 435.7 kB/s eta 0:00:06
    --------------------------------------- 0.0/2.4 MB 435.7 kB/s eta 0:00:06
    --------------------------------------- 0.0/2.4 MB 217.9 kB/s eta 0:00:11
   - -------------------------------------- 0.1/2.4 MB 381.3 kB/s eta 0:00:06
   - -------------------------------------- 0.1/2.4 MB 467.6 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/2.4 MB 448.2 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/2.4 MB 448.2 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/2.4 MB 448.2 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/2.4 MB 448.2 kB/s eta 0:00:05
   ---- ----------------------------------- 0.2/2.4 MB 519

In [33]:
#make sure to set your SerpAPI key. This is required for search-based tools such as SerpAPITool
import os
os.environ["SERPAPI_API_KEY"] = 'xxx'
#create search api key from here: https://serpapi.com/

In [35]:
from langchain.agents import load_tools
from smolagents import CodeAgent, LiteLLMModel, Tool

search_tool = Tool.from_langchain(load_tools(["serpapi"])[0])

# Set your Groq key
import os
os.environ["GROQ_API_KEY"] = "xxx"

# Connect to Groq using the groq/ provider prefix
model_g = LiteLLMModel(
    model_id="groq/openai/gpt-oss-120b", # Or another supported model
    temperature=0.1
)

agent = CodeAgent(tools=[search_tool], model=model_g)

agent.run("Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive experiences.")

ImportError: cannot import name 'load_tools' from 'langchain.agents' (C:\Users\User\anaconda3\Lib\site-packages\langchain\agents\__init__.py)

Code doesn't work because modern versions of LangChain have refactored or moved legacy helper functions like load_tools (often residing in langchain_community.agent_toolkits.load_tools or requiring legacy langchain packages).

## Importing a tool collection from any MCP server
smolagents also allows importing tools from the hundreds of MCP servers available on glama.ai or smithery.ai. If you want to dive deeper about MCP, you can check our free MCP Course.

In [ ]:
# The MCP servers tools can be loaded in a ToolCollection object as follow
import os
from smolagents import ToolCollection, CodeAgent
from mcp import StdioServerParameters
from smolagents import InferenceClientModel


model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")


server_parameters = StdioServerParameters(
    command="uvx",
    args=["--quiet", "pubmedmcp@0.1.3"],
    env={"UV_PYTHON": "3.12", **os.environ},
)

with ToolCollection.from_mcp(server_parameters, trust_remote_code=True) as tool_collection:
    agent = CodeAgent(tools=[*tool_collection.tools], model=model, add_base_tools=True)
    agent.run("Please find a remedy for hangover.")

In [38]:
pip install jupyter-ai

Note: you may need to restart the kernel to use updated packages.
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
   ---------------------------------------- 0.0/134.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.8 kB ? eta -:--:--
   --------- ------------------------------ 30.7/134.8 kB 54.6 kB/s eta 0:00:02
   --------- ------------------------------ 30.7/134.8 kB 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.0.0 requires cryptography<43,>=41.0.5, but you have cryptography 50.0.2 which is incompatible.


In [40]:
import os
print(os.getcwd())

C:\Users\User\AI Agents Huggingface
